In [9]:
import os
import json
import spacy
import pandas as pd
from collections import Counter
from pathlib import Path
from tqdm import tqdm

In [10]:
TARGET_DIR = "E:/Callproject/extracted_json" # Update this to your path


# Load the model
nlp = spacy.load("en_core_web_sm")

In [11]:
def get_file_list(root_path, limit=None):
    """Gathers JSON file paths with an optional limit for testing."""
    print("Scanning directory for files...")
    files = list(Path(root_path).rglob("*.json"))
    
    if limit:
        print(f"Test mode active: Limiting to the first {limit} files.")
        return files[:limit]
    
    return files

def stream_texts(file_list):
    """Generator to yield text from files one by one."""
    for file_path in file_list:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            text_parts = []
            for page in data.get('pages', []):
                for frame in page.get('frames', []):
                    for para in frame.get('paragraphs', []):
                        if para.get('text'):
                            text_parts.append(para['text'])
            
            yield " ".join(text_parts)
        except Exception:
            yield "" 

def process_dataset(root_path, file_limit=None):
    # 1. Get files (with optional limit)
    files = get_file_list(root_path, limit=file_limit)
    total_files = len(files)
    
    counts = {"PERSON": Counter(), "GPE": Counter(), "ORG": Counter()}
    
    # 2. Process with Progress Bar
    print(f"Beginning NER on {total_files:,} files...")
    text_stream = stream_texts(files)
    
    # We disable "parser" and "lemmatizer" to speed up processing
    progress_bar = tqdm(nlp.pipe(text_stream, batch_size=50, disable=["parser", "lemmatizer"]), 
                        total=total_files, 
                        desc="NER Progress",
                        unit="file")

    for doc in progress_bar:
        for ent in doc.ents:
            # Map spaCy labels to our categories
            label = ent.label_
            if label == "PERSON":
                counts["PERSON"][ent.text.strip()] += 1
            elif label in ["GPE", "LOC"]:
                counts["GPE"][ent.text.strip()] += 1
            elif label == "ORG":
                counts["ORG"][ent.text.strip()] += 1
                
    return counts


In [15]:
# --- EXECUTION ---

# RUN TEST: Set file_limit to 1000. Set to None for the full 100k.
results = process_dataset(TARGET_DIR, file_limit=5000)

# 3. Export to CSV (Optional but recommended)
all_data = []
for category, counter in results.items():
    for name, freq in counter.most_common():
        all_data.append({"Category": category, "Entity": name, "Frequency": freq})

df_results = pd.DataFrame(all_data)
df_results.to_csv("ner_results_summary.csv", index=False)
print("\nResults exported to 'ner_results_summary.csv'")

# Display top 10 for quick verification
print(df_results.sort_values("Frequency", ascending=False).head(10))

Scanning directory for files...
Test mode active: Limiting to the first 5000 files.
Beginning NER on 5,000 files...


NER Progress: 100%|██████████████████████████████████████████████████████████████| 5000/5000 [03:52<00:00, 21.54file/s]


Results exported to 'ner_results_summary.csv'
      Category       Entity  Frequency
23225      GPE  Kansas City      10942
23226      GPE          Mo.       6588
23227      GPE         Kas.       1816
23228      GPE     Missouri       1245
23229      GPE        Texas        886
23230      GPE         Ark.        694
23231      GPE       Kansas        612
26322      ORG           AP        590
26323      ORG    Interment        517
23232      GPE   Washington        480
